# 5.11 — Kalman filters

The Kalman gain is the trust coefficient between a noisy prediction and a noisy measurement.

Kalman filters are Gaussian HMMs with linear dynamics. They use Bayesian updating, HMM recursion, and foreshadow particle filters for nonlinear or non-Gaussian systems.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(511)


def kalman_filter(A, H, Q, R, measurements, x0, P0):
    A = np.asarray(A, dtype=float)
    H = np.asarray(H, dtype=float)
    Q = np.asarray(Q, dtype=float)
    R = np.asarray(R, dtype=float)
    x = np.asarray(x0, dtype=float)
    P = np.asarray(P0, dtype=float)
    means = []
    covs = []
    gains = []
    predictions = []
    for z in measurements:
        x_pred = A @ x
        P_pred = A @ P @ A.T + Q
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)
        innovation = np.asarray(z, dtype=float) - H @ x_pred
        x = x_pred + K @ innovation
        P = (np.eye(P.shape[0]) - K @ H) @ P_pred
        means.append(x.copy())
        covs.append(P.copy())
        gains.append(K.copy())
        predictions.append((x_pred.copy(), P_pred.copy()))
    return {"means": np.asarray(means), "covs": np.asarray(covs), "gains": gains, "predictions": predictions}


def kalman_filter_1d(A, H, Q, R, measurements):
    return kalman_filter([[A]], [[H]], [[Q]], [[R]], [[z] for z in measurements], [0.0], [[1.0]])


def simulate_linear_system(A, H, Q, R, x0, steps):
    A = np.asarray(A, dtype=float)
    H = np.asarray(H, dtype=float)
    Q = np.asarray(Q, dtype=float)
    R = np.asarray(R, dtype=float)
    x = np.asarray(x0, dtype=float)
    states = []
    measurements = []
    for t in range(steps):
        process = rng.multivariate_normal(np.zeros(A.shape[0]), Q)
        x = A @ x + process
        obs = H @ x + rng.multivariate_normal(np.zeros(H.shape[0]), R)
        states.append(x.copy())
        measurements.append(obs.copy())
    return np.asarray(states), measurements


def make_kalman_ladder():
    d1_states, d1_meas = simulate_linear_system([[1.0]], [[1.0]], [[0.2]], [[0.5]], [0.0], 2)
    A2 = [[1.0, 1.0], [0.0, 1.0]]
    H2 = [[1.0, 0.0]]
    d2_states, d2_meas = simulate_linear_system(A2, H2, [[0.05, 0.0], [0.0, 0.02]], [[0.4]], [0.0, 1.0], 4)
    d3_states, d3_meas = simulate_linear_system([[1.0]], [[1.0]], [[0.1]], [[0.3]], [0.0], 8)
    d3_meas[3] = d3_meas[3] + np.array([2.5])
    A4 = [[1.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0], [0.0, 0.0, 0.0, 1.0]]
    H4 = [[1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0]]
    d4_states, d4_meas = simulate_linear_system(A4, H4, np.eye(4) * 0.03, np.array([[0.4, 0.2], [0.2, 0.5]]), [0.0, 0.8, 1.0, -0.4], 10)
    d5_states, d5_meas = simulate_linear_system(A2, H2, [[0.08, 0.0], [0.0, 0.04]], [[0.2]], [0.0, 0.4], 14)
    d5_meas = [z + np.array([0.15 * (t ** 1.2)]) for t, z in enumerate(d5_meas)]
    return [
        {"name": "D1 1-D 2-step linear Gaussian", "A": [[1.0]], "H": [[1.0]], "Q": [[0.2]], "R": [[0.5]], "x0": [0.0], "P0": [[1.0]], "states": d1_states, "measurements": d1_meas},
        {"name": "D2 constant velocity", "A": A2, "H": H2, "Q": [[0.05, 0.0], [0.0, 0.02]], "R": [[0.4]], "x0": [0.0, 1.0], "P0": np.eye(2), "states": d2_states, "measurements": d2_meas},
        {"name": "D3 non-Gaussian outlier", "A": [[1.0]], "H": [[1.0]], "Q": [[0.1]], "R": [[0.3]], "x0": [0.0], "P0": [[1.0]], "states": d3_states, "measurements": d3_meas},
        {"name": "D4 correlated 2-D tracking", "A": A4, "H": H4, "Q": np.eye(4) * 0.03, "R": np.array([[0.4, 0.2], [0.2, 0.5]]), "x0": [0.0, 0.8, 1.0, -0.4], "P0": np.eye(4), "states": d4_states, "measurements": d4_meas},
        {"name": "D5 nonlinear mismatch", "A": A2, "H": H2, "Q": [[0.08, 0.0], [0.0, 0.04]], "R": [[0.2]], "x0": [0.0, 0.4], "P0": np.eye(2), "states": d5_states, "measurements": d5_meas},
    ]


## The concept, built once (D1)

The linear Gaussian prediction and update use

$$\hat x_t^-=A\hat x_{t-1},\quad P_t^-=AP_{t-1}A^\top+Q,\quad K_t=P_t^-H^\top(HP_t^-H^\top+R)^{-1}.$$

The lesson predicts variance $P^-=1+0.2=1.200$.

In [ ]:

result = kalman_filter_1d(A=1.0, H=1.0, Q=0.2, R=0.5, measurements=[1.4])
pred_mean, pred_var = result["predictions"][0]
gain = result["gains"][0]
updated_mean = result["means"][0]
updated_var = result["covs"][0]
print("predicted variance", pred_var[0, 0])
print("gain", gain[0, 0])
assert np.isclose(pred_var[0, 0], 1.2)
assert np.isclose(gain[0, 0], 1.2 / (1.2 + 0.5))
assert np.isclose(round(gain[0, 0], 6), 0.705882)


The gain turns the residual into an updated mean, and the posterior variance shrinks because measurement information was absorbed.

In [ ]:

print("updated mean", updated_mean[0])
print("updated variance", updated_var[0, 0])
assert np.isclose(updated_mean[0], 0.9882352941176471)
assert np.isclose(round(updated_mean[0], 6), 0.988235)
assert np.isclose(updated_var[0, 0], 0.35294117647058826)
assert np.isclose(round(updated_var[0, 0], 6), 0.352941)


## The dataset ladder

D1-D5 move from a scalar Gaussian system to velocity, outliers, correlated 2-D tracking, and nonlinear mismatch.

In [ ]:

ladder = make_kalman_ladder()
for rung in ladder:
    print(rung["name"], "steps", len(rung["measurements"]), "state dim", len(rung["x0"]))
    print("measurement sample", [np.round(z, 3).tolist() for z in rung["measurements"][:3]])


In [ ]:

rows = []
for rung in ladder:
    result = kalman_filter(rung["A"], rung["H"], rung["Q"], rung["R"], rung["measurements"], rung["x0"], rung["P0"])
    estimated_positions = result["means"][:, 0]
    true_positions = rung["states"][:, 0]
    rmse = float(np.sqrt(np.mean((estimated_positions - true_positions) ** 2)))
    rows.append({"name": rung["name"], "est": estimated_positions, "true": true_positions, "rmse": rmse, "covs": result["covs"]})
for row in rows:
    print(row["name"], "filtering RMSE", f"{row['rmse']:.6f}")


In [ ]:

fig, axes = plt.subplots(2, len(rows), figsize=(16, 6))
for j, row in enumerate(rows):
    axes[0, j].plot(row["true"], label="true")
    axes[0, j].plot(row["est"], label="filtered")
    axes[0, j].set_title(row["name"].split()[0])
    error_curve = np.sqrt((row["est"] - row["true"]) ** 2)
    axes[1, j].plot(error_curve, marker="o")
    axes[1, j].set_title(f"RMSE {row['rmse']:.3f}")
    axes[1, j].set_xlabel("time")
axes[0, 0].legend()
fig.suptitle("Filtered estimates vs true trajectories and error curve")
plt.tight_layout()


## Pitfall on the hardest rung

Bad $Q,R$ tuning and forgetting process noise causes lag or overconfidence. On D5, a too-small process variance cannot follow the nonlinear measurement drift.

In [ ]:

d5 = ladder[-1]
bad = kalman_filter(d5["A"], d5["H"], [[0.001, 0.0], [0.0, 0.001]], d5["R"], d5["measurements"], d5["x0"], d5["P0"])
good = kalman_filter(d5["A"], d5["H"], d5["Q"], d5["R"], d5["measurements"], d5["x0"], d5["P0"])
bad_rmse = float(np.sqrt(np.mean((bad["means"][:, 0] - d5["states"][:, 0]) ** 2)))
good_rmse = float(np.sqrt(np.mean((good["means"][:, 0] - d5["states"][:, 0]) ** 2)))
print("bad Q RMSE", bad_rmse)
print("calibrated Q RMSE", good_rmse)
print("first predicted variance with Q", good["predictions"][0][1][0, 0])
assert good["predictions"][0][1][0, 0] > d5["P0"][0, 0]


## Evaluate it + Practice

- Metric: filtering RMSE against the simulated true trajectory
- No-skill baseline: raw measurement as the position estimate
- Cheap sanity check: D1 gain equals 0.705882
- Ablation: set Q near zero and watch lag or overconfidence
- Failure signals: large innovations, shrinking covariance despite drift, or jitter from bad R


Practice: Double R in D2 and inspect the gain.

Practice: Inject a D3 outlier and compare RMSE.

Practice: Plot D4 x and y positions separately.